[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/paabL/rl-hvac-lab/blob/main/en/rl_lab_student_en.ipynb)

# Can one thermostat heat a thousand houses it has never seen?

**IFT3700/IFT6758 · Data Science · RL lab · student version**

In Lecture 2, RL was used to align LLMs with human preferences. Today we ask it for **general heating control**: one controller, the same weights for every house, with no model and no tuning for each one. We try it on 1,000 houses in Montreal in January, climbing a ladder with five rungs.

> **This is not a classic RL lab.** We start from textbook RL (one house, one agent) and climb to open research questions: generalizing to buildings never seen, adapting *in context* without changing weights, and keeping on learning when the building changes (meta-RL). It is a glimpse of where research in RL and general HVAC control stands today: the top rungs work in our small simulator, but remain an open problem on real buildings.

> ⚠️ **Run the Setup cell first.** It downloads the companion files (`lab_utils_en.py`, `rl_sb3_en.py`, `gru_ppo_en.py`, `schemas.py`, `images/`, `agents/`, the weather CSV) by itself and installs Stable-Baselines3 if needed. If it cannot reach GitHub, it shows a button to upload **`rl_lab_en.zip`**. Skipping this cell gives `No module named 'lab_utils_en'`.

> **Exercises.** Replace each `___` with your code, then run the cell. While an exercise is empty, the notebook keeps going with the lab's version.

> The weather is **synthetic**. The memoryless agents come from Stable-Baselines3, the memory agents from our small GRU PPO; all are pre-computed on CPU (600 PPO iterations for the SB3 agents, 1,200 for the memory agents, one random seed).

## 0 · Setup

In [ ]:
# Session settings
SHOW_WIDGETS = True    # interactive sliders (False: static figures instead)
RUN_TRAINING = False   # True: retrains the "classic" agent live (~1.5 min on CPU)
REAL_WEATHER = False   # True: replaces the synthetic weather with Montreal Januaries 2015-2024 from Open-Meteo (needs Internet)

# Companion files: lab_utils_en.py, rl_sb3_en.py, gru_ppo_en.py, schemas.py, blanks.py, images/, agents/, montreal_weather.csv
import os, sys, subprocess, urllib.request, zipfile
try:
    import stable_baselines3                         # Stable-Baselines3 (PPO), not in Colab by default
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "stable-baselines3"], check=True)
if not os.path.exists("blanks.py"):
    bundle = "rl_lab_en.zip"
    try:
        urllib.request.urlretrieve("https://raw.githubusercontent.com/paabL/rl-hvac-lab/main/en/rl_lab_en.zip", bundle)
        print("Companion files downloaded from GitHub.")
    except OSError:                                 # GitHub unreachable: ask for the zip
        try:
            from google.colab import files          # only exists in Colab
        except ImportError:
            raise FileNotFoundError("blanks.py not found: open this notebook from the en/ folder of the lab.")
        print("Download failed. Click 'Choose files' below and pick rl_lab_en.zip.")
        zips = [name for name in files.upload() if name.endswith(".zip")]
        assert zips, "That was not a .zip file: please upload rl_lab_en.zip."
        bundle = zips[0]
    archive = zipfile.ZipFile(bundle)
    for name in archive.namelist():
        if not name.endswith(".ipynb"):             # keep this notebook as it is
            archive.extract(name, ".")
sys.path.insert(0, ".")

import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt, warnings
warnings.filterwarnings("ignore")
from lab_utils_en import *
from rl_sb3_en import *
from gru_ppo_en import GRUAgent, train_gru
from schemas import house_diagram, loop_diagram, agent_diagram, preheating_diagram, fleet_diagram
from blanks import ___, check
plt.rcParams.update({"figure.figsize": (10, 4), "font.size": 11, "axes.spines.top": False, "axes.spines.right": False})
torch.set_num_threads(4)

if REAL_WEATHER:
    download_open_meteo().to_csv("montreal_weather.csv", index=False)
weather = load_weather()
print(f"{len(weather)} hours of January weather, from {weather.t_ext.min():.0f} to {weather.t_ext.max():.0f} °C")

### What is already coded

Students only write the essentials, in five exercises: the physics, REINFORCE, drawing houses, the saddle gradient and the half-life. The rest is provided, and we rely on standard libraries (Gymnasium, Stable-Baselines3) wherever the code details add nothing. Only the memory PPO is in-house: no library offers the prediction head.

| File | Function or class | What it does | Returns |
| --- | --- | --- | --- |
| `lab_utils_en.py` | `load_weather()` | Montreal Januaries, hourly | a DataFrame |
| | `sample_houses(n, width)` | draws n random houses in the parameter box (exercise 3) | dict `UA`, `Cm`, `mu` |
| | `thermal_step(Ta, Tm, u, t_ext, xi)` | one hour of the house model (exercise 1) | `Ta`, `Tm`, kWh |
| | `Houses(weather, n)` | N houses simulated at once: `reset()`, `step(actions)` | observations, rewards, costs |
| | `thermostat(env)`, `preheating_thermostat(env)` | the two baseline controllers, no learning | actions |
| | `reinforce(weather, update=...)` | REINFORCE on one house, with your update (exercise 2) | weights, curve |
| | `J(b, c)`, `grad_J(b, c)` | the saddle toy (exercise 4) | cost, gradient |
| `rl_sb3_en.py` | `HouseGym(weather)` | one house following the Gymnasium convention (below) | a `gymnasium.Env` |
| | `HousesVec(weather, n)` | 256 houses at once, as a Stable-Baselines3 `VecEnv` | to train fast |
| | `make_agent(weather)` | an SB3 `PPO` without memory (an MLP) | an SB3 agent |
| | `load_agent(name)` | a pre-trained agent from the `agents/` folder | agent, history |
| | `evaluate(policy, weather, xi, starts)` | plays one week without exploration, on fixed houses | `cost`, `discomfort`, `Ta`, memory `m` |
| | `test_bench(weather, n)`, `cost_per_day(d)` | n test houses and weeks, the same for everyone; the mean cost per day | |
| | `erase_effect`, `swap_effect`, `probe` | the three memory experiments (Section 5) | extra cost in %, $R^2$ |
| `gru_ppo_en.py` | `GRUAgent(prediction_head)` | our memory agent: a GRU, an actor, a critic, a prediction head | a PyTorch network |
| | `train_gru(agent, houses, iterations)` | our memory PPO, short version (to call, not to read) | learning curve |
| `schemas.py` | `..._diagram` | the notebook's diagrams | a figure |
| `blanks.py` | `___`, `check` | the exercise blanks and their checks | |

### The environment: the Gym convention

In RL, an **environment** almost always follows the [Gymnasium](https://gymnasium.farama.org) convention (formerly OpenAI Gym):

- `observation_space` and `action_space` describe what the agent sees and what it can do. Here: 9 numbers (indoor and outdoor temperature, price, upcoming peak, hour, last action) and `Discrete(5)`, five power levels.
- `reset()` starts an episode and returns `(observation, info)`.
- `step(action)` advances one hour and returns `(observation, reward, terminated, truncated, info)`. `terminated` flags a natural end (here, never); `truncated`, an end by time limit (here, after 168 hours).

`HouseGym` is one house following this convention: any RL library can use it. To train, `HousesVec` advances 256 houses at once, as a Stable-Baselines3 `VecEnv`. Here is the agent-environment loop, with an agent that plays at random:

In [ ]:
env = HouseGym(weather)                        # one random house
obs, info = env.reset(seed=0)
total = 0.0
while True:
    action = env.action_space.sample()         # an agent that plays at random
    obs, reward, terminated, truncated, info = env.step(action)
    total += reward
    if terminated or truncated:
        break
print(env.observation_space, env.action_space)
print(f"One random week: {-total:.0f} $ of cost")

## 1 · Why control heating?

- **The stake.** Buildings in operation emitted 9.9 Gt of CO₂ in 2024, 26% of energy-related CO₂ (UNEP, *Global Status Report* 2026).
- **The lever.** Heating, ventilation and air conditioning (HVAC) are 38% of building energy (González-Torres et al., 2022).
- **The margin.** Reviews report 13 to 28% savings from better control (Drgoňa et al., 2020).
- **The peak.** In Quebec, on winter mornings and evenings, everyone heats at the same time. Hydro-Québec offers peak-time rates to spread demand: heat *before* the peak rather than during it.

Preheating means storing heat in the walls before the peak. But the right plan depends on the house:

In [ ]:
preheating_diagram("en"); plt.show()

Model predictive control (MPC) does this job very well... provided there is a model of each building. Building and calibrating that model house by house is too costly for millions of homes. Hence our through-line: **a general controller**. Can RL learn a single controller that adapts to each house on its own?

In [ ]:
fleet_diagram("en"); plt.show()

### The five regimes of the ladder

We never change the agent, only the world it trains in. At each rung it has to get a little smarter.

| Regime | In one sentence | Training world | Section |
| --- | --- | --- | --- |
| Classic RL | learn to control **one** given house | a single house | 3 |
| Robust | find **one compromise**, the same for every house | houses drawn at random, no memory | 4 |
| Memorize or recognize | recognize a house **already seen** in training | a few houses, with memory | 6 |
| In-context learning | estimate a house **never seen** by watching it, without changing its weights | a new house every episode | 6 |
| Meta-RL | **keep learning** when the house changes along the way | houses that change during the episode | 7 |

**Meta-RL in plain words: learning to learn.** During training the agent sees so many houses, and houses that change, that it learns a *way* of adapting. Once deployed, its weights no longer move: its memory learns the new house hour after hour, and relearns it if it changes.

## 2 · The problem: heating without paying peak prices

> **RL vocabulary** (new in this course)
> - **State** $s_t$: what the agent observes at hour $t$ (indoor and outdoor temperature, price, time of day).
> - **Action** $a_t$: the heat-pump power, from 0 to 100% in steps of 25%.
> - **Reward** $r_t = -\text{cost}_t$, with $\text{cost}_t = \text{price}_t \times \text{kWh}_t + \lambda \times \text{discomfort}_t$ (degree-hours outside 20-23 °C, $\lambda$ = 1 \$ per K·h).
> - **Return** $G_t = \sum_{k \ge t} r_k$: everything left to gain until the end of the week. The agent maximizes the return, not the next reward: paying a little now (preheating) can save more later.
> - **Policy** $\pi(a \mid s)$: the decision rule, the thing we learn.
> - **Episode**: one week, 168 hourly decisions, in one house.

In [ ]:
loop_diagram("en"); plt.show()

**The house** has two temperatures: the air $T_a$, which reacts within minutes, and the thermal mass $T_m$ (walls, floors), which reacts over hours and is **invisible** to the thermostat. Each follows a heat balance, (heat capacity) × (rate of change) = sum of heat flows:

$$C_a \frac{dT_a}{dt} = UA\,(T_\text{ext} - T_a) + H_{am}\,(T_m - T_a) + Q_\text{pump} + Q_\text{int}, \qquad C_m \frac{dT_m}{dt} = H_{am}\,(T_a - T_m)$$

Three parameters $\xi$ change from house to house, and the agent never sees them: the **insulation** $UA$, the **thermal mass** $C_m$ and the **heat-pump size** $\mu$. Electricity is at peak price on weekdays from 6 to 9 am and 4 to 8 pm (0.50 \$/kWh instead of 0.07).

In [ ]:
house_diagram("en"); plt.show()

In [ ]:
# EXERCISE 1/5 · One hour of the two-temperature model (Euler method, 6 substeps of 10 min)
def my_thermal_step(Ta, Tm, u, t_ext, xi, substeps=6):
    q_pump = u * xi["mu"] * xi["UA"] * (T_SETPOINT - T_DESIGN)     # heat delivered (W)
    dt = 3600.0 / substeps                                          # seconds
    for _ in range(substeps):
        # TO DO: heat balance of the air (loss, exchange with the mass, pump, internal gains), then of the mass
        # Available: xi["UA"], xi["Cm"], H_AM, C_AIR, Q_INT, q_pump, t_ext
        dTa = ___
        dTm = ___
        Ta, Tm = Ta + dt * dTa, Tm + dt * dTm
    return Ta, Tm, q_pump / cop(t_ext) / 1000.0                     # electricity used (kWh) = heat / COP

xi_check = sample_houses(5, rng=np.random.default_rng(0))

def test_exercise_1():
    return np.allclose(my_thermal_step(20.0, 19.0, 0.5, -10.0, xi_check), thermal_step(20.0, 19.0, 0.5, -10.0, xi_check))

done = check("Exercise 1", test_exercise_1)

Here are 64 random houses over one week, all heated by the same proportional thermostat set to 21 °C. Grey bands are peak hours.

In [ ]:
xi_test, starts_test = test_bench(weather, n=256)      # 256 test houses and weather weeks, the same for every agent
d_th = evaluate(thermostat, weather, xi_test, starts_test)
peak = is_peak(np.arange(H))                           # 1 at peak hours, 0 otherwise
fig, ax = plt.subplots(figsize=(11, 4))
im = ax.imshow(d_th["Ta"][:64], aspect="auto", cmap="coolwarm", vmin=17, vmax=26)
for t in range(H):
    if peak[t] == 1:
        ax.axvspan(t - 0.5, t + 0.5, color="k", alpha=0.12, lw=0)
ax.set(xlabel="hour of the week", ylabel="house", title="The same thermostat in 64 houses: smooth in some, wild swings in others")
plt.colorbar(im, label="indoor temperature (°C)"); plt.show()
print(f"Mean cost: {cost_per_day(d_th).mean():.2f} $/day; from {cost_per_day(d_th).min():.2f} to {cost_per_day(d_th).max():.2f} depending on the house")

## 3 · Classic RL: a single house

> **Reminder: Lecture 2.** Logistic regression is trained by $w_j \leftarrow w_j + \eta \sum_i (y_i - \hat y_i)\, x^i_j$.

> **The policy gradient.** We look for the weights $w$ that maximize the mean return $J(w) = \mathbb{E}_{\pi_w}[G_0]$. The policy-gradient theorem gives
> $$\nabla_w J = \mathbb{E}\Big[\sum_t G_t\, \nabla_w \log \pi_w(a_t \mid s_t)\Big].$$
> Our policy is a logistic regression on 7 features $x_t$: $\pi_w(\text{heat} \mid s_t) = \sigma(w^\top x_t)$. Then $\nabla_w \log \pi_w(a_t \mid s_t) = \big(a_t - \sigma(w^\top x_t)\big)\, x_t$, hence the **REINFORCE** update:
> $$w_j \leftarrow w_j + \eta \sum_t G_t\, \big(a_t - \sigma(w^\top x_t)\big)\, x_{t,j}.$$
> This is the logistic regression of Lecture 2: the label is the action taken, and each example is weighted by the return that followed it. Actions followed by a good return become more likely.

> **The baseline.** We replace $G_t$ by $G_t - b_t$, with $b_t$ the mean return at the same hour across houses. On average the gradient does not change (because $\mathbb{E}[\nabla \log \pi] = 0$), but it gets much less noisy.

In [ ]:
# EXERCISE 2/5 · The REINFORCE update
def my_update(w, X, A, P, G, eta):
    # X: (hours, houses, 7) features; A: actions taken, 0 or 1; P: probabilities π(heat); G: returns minus the baseline
    # TO DO: the logistic regression of Lecture 2, each example weighted by its return
    X2 = X.reshape(-1, X.shape[-1])             # one example per row
    weights = ___                               # return × (action - probability), one number per example
    gradient = ___                              # like logistic regression: Xᵀ (y - ŷ) / n
    return w + eta * gradient

def test_exercise_2():
    r = np.random.default_rng(0)
    X = r.normal(size=(4, 3, 7))
    A = r.integers(0, 2, (4, 3))
    P = r.uniform(size=(4, 3))
    G = r.normal(size=(4, 3))
    return np.allclose(my_update(np.zeros(7), X, A, P, G, 0.1), reinforce_update(np.zeros(7), X, A, P, G, 0.1))

done = check("Exercise 2", test_exercise_2, fallback=True)
if done:
    w, hist = reinforce(weather, iterations=100, eta=5.0, update=my_update, verbose=False)
else:
    w, hist = reinforce(weather, iterations=100, eta=5.0, verbose=False)     # the lab's version
names = ["bias", "Ta - 21", "T_ext / 10", "peak now", "peak soon", "sin hour", "cos hour"]
print(pd.Series(w, index=names).round(2).to_string())
plt.plot(hist); plt.yscale("log"); plt.xlabel("iteration"); plt.ylabel("cost ($/day)"); plt.title("REINFORCE learns to heat in a few seconds"); plt.show()

**Question:** read the weights as you would for a logistic regression. Why is the weight of "Ta - 21" strongly negative? And the weight of "peak now"? Why does "peak soon" stay close to 0?

### Policy gradient is only one family among several

| Family | What is learned | Examples | In this lab |
| --- | --- | --- | --- |
| Policy gradient | the policy $\pi$ directly | REINFORCE, PPO | REINFORCE here; PPO for every later agent |
| Actor-critic | $\pi$ and a value $V$ used as a baseline | A2C, PPO, SAC | our PPO agents: an actor, a critic |
| Value-based | the value $Q(s, a)$ of each action, then take the best | Q-learning, DQN | — |
| Model-based | a model of the dynamics, then plan | MPC, Dyna, MuZero | MPC, played here by the oracle |
| Imitation | copy an expert | behavior cloning, DAgger | — |

> **From REINFORCE to PPO: the policy gradient with advantages.** Three changes to the REINFORCE formula, for the weights $\theta$ of a network.
>
> **1. The advantage instead of the return.** A critic $V(s)$ learns to predict the return. The **advantage** $A_t = Q(s_t, a_t) - V(s_t)$ says whether the action did better or worse than expected; we estimate it by $\hat A_t = G_t - V(s_t)$. It is the baseline, but learned:
> $$\nabla_\theta J = \mathbb{E}\Big[\sum_t \hat A_t\, \nabla_\theta \log \pi_\theta(a_t \mid s_t)\Big].$$
>
> **2. A less noisy estimate (GAE).** Instead of waiting for the end of the week, each hour is compared with what the critic expected: $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$, then $\hat A_t = \delta_t + \gamma\lambda\,\delta_{t+1} + (\gamma\lambda)^2\,\delta_{t+2} + \dots$ (here $\gamma = 0.97$ and $\lambda = 0.95$).
>
> **3. The safeguard (clipping).** PPO reuses each batch of experience several times. To avoid drifting too far from the policy that collected it, it maximizes
> $$L^{\text{clip}}(\theta) = \mathbb{E}_t\Big[\min\big(\rho_t\,\hat A_t,\ \operatorname{clip}(\rho_t,\, 1-\varepsilon,\, 1+\varepsilon)\,\hat A_t\big)\Big], \qquad \rho_t = \frac{\pi_\theta(a_t \mid s_t)}{\pi_{\theta_\text{old}}(a_t \mid s_t)}, \quad \varepsilon = 0.2.$$
> At the start of each update, $\rho_t = 1$ and this gradient is exactly the one of point 1. As soon as the ratio $\rho_t$ leaves $[0.8, 1.2]$ in the improving direction, the gradient vanishes: one update cannot change the policy too much.
>
> The full loss adds the critic's error, $(V(s_t) - G_t)^2$, and a small entropy bonus to keep exploring. We do not code PPO: for memoryless agents we use **Stable-Baselines3** (SB3). For memory agents, `gru_ppo_en.py` holds a short version of the same algorithm, with the prediction head that libraries do not offer.

An on/off agent is crude. With five power levels, a memory (a GRU) and PPO, we get the "classic" agent. Trained on this single house, it learns to preheat before the peak. Training it takes two lines and about 1.5 minutes on CPU:

In [ ]:
if RUN_TRAINING:
    classic_agent = GRUAgent()                                                   # our memory agent (gru_ppo_en.py)
    train_gru(classic_agent, Houses(weather, 256, xi=nominal_house(256)), iterations=60)   # 256 copies of the house
else:
    classic_agent, _ = load_agent("classic")

xi_nom = nominal_house(256)
for name, pol in [("thermostat", thermostat), ("preheating thermostat", preheating_thermostat), ("classic agent", classic_agent)]:
    print(f"{name:28s} {cost_per_day(evaluate(pol, weather, xi_nom, starts_test)).mean():6.2f} $/day")

fig, ax = plt.subplots(figsize=(11, 3.5))
for name, pol in [("thermostat", thermostat), ("classic agent", classic_agent)]:
    ax.plot(evaluate(pol, weather, xi_nom, starts_test)["Ta"][0, :72], label=name)
for t in range(72):
    if peak[t] == 1:
        ax.axvspan(t - 0.5, t + 0.5, color="k", alpha=0.1, lw=0)
ax.axhspan(*COMFORT_BAND, color="g", alpha=.08); ax.set(xlabel="hour", ylabel="°C", title="The agent warms up before the peak and coasts down during it"); ax.legend(); plt.show()

> **On a known house, RL is useless: MPC is enough.** If the model of the house is known (its parameters $\xi$), model predictive control directly optimizes the next hours from the weather and price forecasts, with no training at all. In our numbers, the **oracle**, an agent that is given $\xi$, plays that role. RL becomes interesting when we want **one controller for every house**, with no model of each.

> **Towards general control.** The classic agent knows a single house: it is not general at all. Let us see what it does elsewhere.

## 4 · Robust: one compromise for all houses

To drive a thousand houses, we train on houses drawn at random: this is the **data augmentation** of Lectures 1 and 7, called *domain randomization* in RL. A memoryless agent (an MLP that only sees the current observation, trained with SB3's PPO) cannot tell houses apart, so the best it can learn is one compromise that works reasonably everywhere.

In [ ]:
# EXERCISE 3/5 · Draw n houses: ξ = nominal × exp(U(-w, w)), w = width × log(sqrt(factor))
def my_houses(n, width=1.0, rng=np.random.default_rng(0)):
    # TO DO: for each parameter k, n values using NOMINAL[k] and FACTOR[k]
    houses = {}
    for k in NOMINAL:
        w = ___
        houses[k] = ___
    return houses

def test_exercise_3():
    mine = my_houses(5, rng=np.random.default_rng(0))
    lab = sample_houses(5, rng=np.random.default_rng(0))
    for k in NOMINAL:
        if not np.allclose(mine[k], lab[k]):
            return False
    return True

done = check("Exercise 3", test_exercise_3, fallback=True)
if done:
    xi_demo = my_houses(2000, rng=np.random.default_rng(0))
else:
    xi_demo = sample_houses(2000, rng=np.random.default_rng(0))             # the lab's version
for k in xi_demo:
    print(k, "from", round(xi_demo[k].min(), 2), "to", round(xi_demo[k].max(), 2))

In [ ]:
oracle, _ = load_agent("oracle")      # is told ξ and the mass temperature: the yardstick, not a real controller
robust, _ = load_agent("robust")      # SB3 PPO without memory, trained on random houses
res = {}
res["thermostat"] = cost_per_day(d_th)
res["classic (one house)"] = cost_per_day(evaluate(classic_agent, weather, xi_test, starts_test))
res["robust (no memory)"] = cost_per_day(evaluate(robust, weather, xi_test, starts_test))
res["oracle (sees ξ)"] = cost_per_day(evaluate(oracle, weather, xi_test, starts_test, privileged=True))
for k, v in res.items():
    print(f"{k:26s} {v.mean():6.2f} $/day   gap to oracle {100 * (v.mean() / res['oracle (sees ξ)'].mean() - 1):+6.1f} %")

gap = 100 * (res["robust (no memory)"] / res["oracle (sees ξ)"] - 1)
plt.figure(figsize=(6, 4.5))
sc = plt.scatter(xi_test["UA"], xi_test["mu"], c=gap, cmap="magma_r", s=25)
plt.colorbar(sc, label="extra cost of robust vs oracle (%)")
plt.xscale("log"); plt.yscale("log"); plt.xlabel("insulation UA (W/K, higher = leakier)"); plt.ylabel("heat-pump size μ")
plt.title(f"The price of not knowing: the same compromise\ncosts up to +{gap.max():.0f}% depending on the house"); plt.show()

> **Towards general control.** One controller for every house: done, at about +11% above the oracle. But it is blind to the house. To do better, it would have to recognize it.

## 5 · Does the agent listen to its memory?

> **Why a memory can recognize the house.** Heating, then watching how the temperature responds, is what engineers do to model a building: **system identification**. A leaky house cools fast when the heat is cut; a small heat pump warms slowly when pushed. With a few hours of history (power requested, temperatures), a simple regression would estimate $UA$, $C_m$ and $\mu$. A memory can do this computation implicitly, without being told what to look for... if training pushes it to.

So we give the agent a memory: a **GRU**, which you can read as an MLP that reads its own previous output every hour, $h_t = f(W[x_t, h_{t-1}] + w_0)$. On top of it, three heads.

In [ ]:
agent_diagram("en"); plt.show()

> **Actor-critic.** The actor picks the action. The critic $V(h_t)$ predicts the expected return and serves as a baseline: the actor learns from the **advantage** $A_t = G_t - V(h_t)$, "better or worse than expected". It is the advantage of the PPO formula (Section 3): here the critic reads the memory $h_t$.

> **Self-predictive RL.** The reward says whether the action was good, not what the house is: a weak, noisy signal for learning to identify it. The third head adds a self-supervised task: predict next hour's temperature from the memory and the action. The target is free, it is the thermometer. To predict well, the memory must contain the house. Making the agent predict its own future helps a lot when the state is hidden (Schwarzer et al., 2021 (UdeM, Mila); Ni et al., 2024 (UdeM, Mila)). Here its loss is simply added to PPO's, in `gru_ppo_en.py`.

A toy with two weights shows why RL alone struggles. The memory writes $m = c\,\xi + \varepsilon$ (what it stores about the house, plus noise), the action reads $a = b\,m$, and the ideal action is $\xi$:

$$J(b, c) = \sigma_\xi^2\,(bc - 1)^2 + b^2 \sigma_\varepsilon^2$$

Here $\sigma_\xi$ measures how different the houses are and $\sigma_\varepsilon$ the noise. At $(0, 0)$, the robust policy, reading brings nothing while the memory is empty, and writing brings nothing while nobody reads: the gradient is zero. $(0, 0)$ is a **saddle point**, and gradient descent can linger there.

In [ ]:
# EXERCISE 4/5 · The gradient of the toy
def my_grad_J(b, c, s_xi=1.0, s_eps=1.0):
    # TO DO: dJ/db, dJ/dc
    error = b * c - 1
    dJ_db = ___
    dJ_dc = ___
    return dJ_db, dJ_dc

def test_exercise_4():
    return np.allclose(my_grad_J(.3, -.7), grad_J(.3, -.7))

done = check("Exercise 4", test_exercise_4, fallback=True)
if done:
    gradient = my_grad_J
else:
    gradient = grad_J                                                       # the lab's version
print("Gradient at the saddle (0, 0):", gradient(0.0, 0.0))

def landscape(s_xi=1.0, head=0.0):
    grid = np.linspace(-1.5, 1.5, 200)
    B, Cc = np.meshgrid(grid, grid)
    shown_cost = J(B, Cc, s_xi) + head * (1 - Cc) ** 2
    plt.figure(figsize=(5.5, 5))
    plt.contourf(B, Cc, np.log(shown_cost + 1e-3), 30, cmap="Blues_r")
    rng = np.random.default_rng(1)
    for _ in range(20):
        b, c = rng.normal(0, 1e-3, 2)        # start very close to the saddle
        traj = [(b, c)]
        for _ in range(150):
            gb, gc = gradient(b, c, s_xi)
            gc += -2 * head * (1 - c)        # the prediction head: a gradient for writing that does not go through reading
            b, c = b - 0.03 * gb, c - 0.03 * gc
            traj.append((b, c))
        traj = np.array(traj)
        plt.plot(traj[:, 0], traj[:, 1], "w-", lw=1)
        plt.plot(traj[-1, 0], traj[-1, 1], "wo", ms=4)
    b_target = s_xi**2 / (s_xi**2 + 1.0)       # memory written (c = 1), read as well as possible
    plt.plot(0, 0, "r*", ms=15, label="saddle: robust policy (memory neither written nor read)")
    plt.plot(b_target, 1.0, "P", color="#2ecc40", mec="k", ms=15, label="target: the memory holds the house, the action reads it")
    plt.xlim(-1.5, 1.5); plt.ylim(-1.5, 1.5)
    plt.xlabel("b: read the memory"); plt.ylabel("c: write the house into memory")
    plt.title(f"σξ = {s_xi:.1f}, prediction head = {head:.1f}")
    plt.legend(loc="lower left", fontsize=8.5)
    plt.show()

if SHOW_WIDGETS:
    import ipywidgets as W
    W.interact(landscape, s_xi=W.FloatSlider(1.0, min=0.3, max=2.0, step=0.1, description="width σξ"),
               head=W.FloatSlider(0.0, min=0.0, max=1.0, step=0.1, description="head"))
else:
    landscape(1.0, 0.0); landscape(1.0, 0.5)

The **green cross** is where we want to go: the adaptive policy, where the memory holds the house ($c \neq 0$) and the action uses it ($b \neq 0$). Its cost is lower than the saddle's, because the action fits each house instead of a compromise. The **red star** is where training without the head gets stuck: reading an empty memory is useless, and writing a memory that nobody reads is useless too.

**Question:** widening the box of houses ($\sigma_\xi$) deepens the valley. Does it make the runs leave the saddle faster? What changes when the prediction head is switched on?

Same experiment on the real agents: a GRU trained by PPO **without** a prediction head, and the same **with** the head.

In [ ]:
saddle, _ = load_agent("saddle")
in_context, _ = load_agent("in_context")
rows = []
for name, ag in [("GRU, no head (saddle)", saddle), ("GRU + head", in_context)]:
    d = evaluate(ag, weather, xi_test, starts_test)
    res[name] = cost_per_day(d)
    row = {"agent": name, "cost": res[name].mean()}
    row["erase_memory"] = erase_effect(ag, weather, xi_test, starts_test)
    row["swap_memory"] = swap_effect(ag, weather, xi_test, starts_test)
    r2 = probe(d, (6, 24, 72, 167))
    for t in r2:
        row[f"probe R2 h{t}"] = r2[t]
    rows.append(row)
tab = pd.DataFrame(rows).set_index("agent").round(2)
tab

Each column is an experiment on the agent's memory:
- **cost**: \$/day on the 256 test houses.
- **erase_memory**: wipe the memory at hour 72; extra cost over the next 24 hours (%).
- **swap_memory**: swap the memories of two houses at hour 72, so each agent "believes" it is in the other house; extra cost over the next 24 hours (%).
- **probe R2**: a linear probe (a scikit-learn Ridge regression) reads $\xi$ from the memory; $R^2$ on houses the probe never saw. It measures what the memory *knows*, not whether the policy *uses* it.

**Question:** which agent really *listens to* its memory? Which one knows something without using it?

## 6 · Memorize or recognize, then in-context learning

> **Reminder: Lectures 3 and 4.** Overfitting means memorizing the training set instead of understanding it.

Same agent (GRU + head), three training worlds: 4 houses, 16 houses, or a new house every episode. We test it on its training houses and on houses it has never seen.

In [ ]:
rows = []
for name, k in [("memorize_4", 4), ("memorize_16", 16), ("in_context", None)]:
    ag, _ = load_agent(name)
    if k:
        xi_k = sample_houses(k, 1.0, np.random.default_rng(11))      # the same k houses it was trained on
        idx = np.arange(256) % k
        xi_train = {key: v[idx] for key, v in xi_k.items()}
    else:
        xi_train = sample_houses(256, 1.0, np.random.default_rng(99))
    ref_train = cost_per_day(evaluate(oracle, weather, xi_train, starts_test, privileged=True)).mean()
    c_train = cost_per_day(evaluate(ag, weather, xi_train, starts_test)).mean()
    if k:
        c_test = cost_per_day(evaluate(ag, weather, xi_test, starts_test)).mean()
        world = f"{k} houses"
    else:
        c_test = res["GRU + head"].mean()
        world = "always new"
    rows.append({"world": world,
                 "training houses": 100 * (c_train / ref_train - 1),
                 "new houses": 100 * (c_test / res["oracle (sees ξ)"].mean() - 1)})
gen = pd.DataFrame(rows).set_index("world").round(1)
gen.plot.bar(rot=0, figsize=(8, 3.5), ylabel="gap to oracle (%)", title="Few houses: it memorizes. Many: it generalizes")
plt.show(); gen

**In-context learning.** The weights are now frozen: only the memory changes as the agent watches the house. Each dot is a test house, placed by the first two principal components of the agent's 64-number memory and colored by its insulation.

In [ ]:
from sklearn.decomposition import PCA
d = evaluate(in_context, weather, xi_test, starts_test)
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
for ax, t in zip(axs, (2, 96)):
    p = PCA(2).fit_transform(d["m"][:, t])
    s = ax.scatter(p[:, 0], p[:, 1], c=np.log(xi_test["UA"]), cmap="viridis", s=15)
    ax.set(title=f"Memory at hour {t}", xlabel="PC1", ylabel="PC2")
plt.colorbar(s, ax=axs, label="log insulation UA"); fig.suptitle("Reading the memory: houses line up by insulation as the hours go by", y=1.04); plt.show()

> **Towards general control.** The agent identifies a house it has never seen within a few hours, without touching its weights, like an LLM picking up a task from the examples in its prompt. One case is left: the house that changes.

## 7 · Meta-RL: when the house changes mid-episode

So far a house stays the same for the whole week. Real houses change: a heat pump fails, a window is left open. An agent that identified the house once must keep re-identifying it.

Why that is hard: an estimate that averages **all** of the past reacts more and more slowly. After a jump, it takes as long to correct halfway as its age at the time of the jump: that is its **half-life**. An estimate that forgets (an exponentially weighted mean) keeps a constant half-life.

In [ ]:
# EXERCISE 5/5 · The half-life of an estimate after a jump
def half_life(t0, method, n=400):
    x = pd.Series(np.r_[np.zeros(t0), np.ones(n - t0)])        # the parameter jumps from 0 to 1 at hour t0
    # TO DO: cumulative mean or exponential mean of x (half-life 12 h), using pandas
    if method == "cumulative":
        estimate = ___
    else:
        estimate = ___
    for h in range(n - t0):                                   # first hour after the jump where the estimate passes 0.5
        if estimate[t0 + h] >= 0.5:
            return h

def test_exercise_5():
    return half_life(96, "cumulative") == 95 and half_life(192, "exponential") == 11

done = check("Exercise 5", test_exercise_5)
if done is not None:
    for t0 in [12, 24, 48, 96, 192]:
        print(f"jump at hour {t0:3d}: cumulative half-life {half_life(t0, 'cumulative'):3d} h, exponential {half_life(t0, 'exponential'):3d} h")

In the simulator, the drift strikes at a random hour: the heat pump loses half its power, or the insulation degrades by a factor of 1.8. We compare the agent trained without drift to the one trained with drift.

In [ ]:
drift_agent, _ = load_agent("drift")
oracle_d, _ = load_agent("oracle_drift")
env_d = Houses(weather, 256, drift=True, seed=123); env_d.reset(xi=xi_test, starts=starts_test)   # same jumps as in evaluate()
t_jump = env_d.t_jump
def cost_after_jump(d):
    """Mean cost per day in the 48 hours after each house's jump."""
    total = 0.0
    for i in range(len(t_jump)):
        total += d["cost"][i, t_jump[i]:t_jump[i] + 48].sum()
    return total / len(t_jump) / 2

drift_agents = [("GRU + head, trained without drift", in_context, False),
                ("GRU + head, trained with drift", drift_agent, False),
                ("oracle (sees ξ live)", oracle_d, True)]
week, after, traces = {}, {}, {}
for name, ag, priv in drift_agents:
    d = evaluate(ag, weather, xi_test, starts_test, privileged=priv, drift=True)
    week[name] = cost_per_day(d).mean()
    after[name] = cost_after_jump(d)
    traces[name] = d["Ta"]
ref = after["oracle (sees ξ live)"]
drift_tab = pd.DataFrame({"cost over the week ($/day)": week, "cost in the 48 h after the jump ($/day)": after})
drift_tab["gap to oracle after the jump (%)"] = 100 * (drift_tab["cost in the 48 h after the jump ($/day)"] / ref - 1)
drift_tab.round(2)

In [ ]:
# one house whose heat pump loses half its power, picked where the gap between the two agents is clearest
names = list(traces)

def cold_after_jump(Ta, i):
    """K·h below 20 °C in the 48 hours after house i's jump."""
    return np.maximum(COMFORT_BAND[0] - Ta[i, t_jump[i]:t_jump[i] + 48], 0).sum()

i, best_gap = None, -1.0
for j in range(len(t_jump)):
    pump_broken = env_d.jump_type[j] == 0
    oracle_warm = cold_after_jump(traces[names[2]], j) < 2
    if pump_broken and oracle_warm:
        gap_j = cold_after_jump(traces[names[0]], j) - cold_after_jump(traces[names[1]], j)
        if gap_j > best_gap:
            i, best_gap = j, gap_j

def gap_half_life(Ta, Ta_ref, t0, smooth=6):
    """Hours after the jump until the temperature gap to the oracle falls below half its maximum."""
    g = pd.Series(Ta_ref - Ta).rolling(smooth, min_periods=1).mean().values[t0:]
    top = int(g.argmax())
    below = np.where(g[top:] < g[top] / 2)[0]
    return top + int(below[0]) if len(below) else len(g), g[top]

fig, ax = plt.subplots(figsize=(11, 4))
for name, Ta in traces.items():
    ax.plot(Ta[i], label=name, lw=1.2)
t0 = t_jump[i]
ax.axvline(t0, color="r", ls="--", label="heat pump loses 50%"); ax.axhspan(*COMFORT_BAND, color="g", alpha=.08)
low = min(traces[name][i].min() for name in names)
ax.set_ylim(low - 1.3, None)
for k in [0, 1]:
    name, y = names[k], low - 0.45 - 0.5 * k
    n12, top = gap_half_life(traces[name][i], traces[names[2]][i], t0)
    if top > 0.2:
        ax.annotate("", xy=(t0 + n12, y), xytext=(t0, y), arrowprops=dict(arrowstyle="<->", color=f"C{k}", lw=1.5))
        ax.text(t0 + n12 + 2, y, f"half-life ≈ {n12} h", va="center", color=f"C{k}", fontsize=10)
    else:
        ax.text(t0 + 2, y, "negligible gap to the oracle", va="center", color=f"C{k}", fontsize=10)
ax.set(xlabel="hour", ylabel="°C", title="After the jump: who adapts, and how fast?"); ax.legend(fontsize=9, loc="upper right"); plt.show()

The **half-life** can be read on the figure: the time after the jump for the temperature gap to the oracle to fall back to half its maximum.

**Question:** compare with Exercise 5. Which agent behaves like the cumulative mean, and which like the exponential one?

## 8 · Wrap-up: where each agent stopped

In [ ]:
summary = pd.DataFrame({"cost ($/day)": {k: v.mean() for k, v in res.items()}})
summary["gap to oracle (%)"] = 100 * (summary["cost ($/day)"] / res["oracle (sees ξ)"].mean() - 1)
summary.sort_values("cost ($/day)", ascending=False).round(2)

| Regime | Training world | Agent of the day | What it can do |
| --- | --- | --- | --- |
| Classic RL | one house | classic agent | beats the thermostat at home, fails elsewhere |
| Robust | random houses | PPO without memory; GRU without head | one compromise at +11% above the oracle; +6% for the GRU without head, which barely reads its memory |
| Memorize or recognize | 4 or 16 houses | GRU + head | with 4 houses: +3.5% at home, +15% elsewhere; with 16: +2% and +2% |
| In-context learning | a new house every episode | GRU + head | identifies a house never seen, at +3% above the oracle |
| Meta-RL | the house changes during the episode | GRU + head, trained with drift | after a failure, +7% above the oracle, against +13% without drift in training |

**General control, in short.** An MPC needs the model of each house; our last agent needs only its history. At full scale, the NOMAD research project behind this lab stayed near the saddle: the memory knew a little, but the policy did not read it. Making this work on real buildings is still an open question.